# Heatmap 3

In [ ]:
import os
from datetime import datetime
import matplotlib
import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
from matplotlib import font_manager
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
import NLProcessing

date = datetime.today().strftime('%Y%m%d')

laptop = "C:\\Users\\lnico\\ACC Lab Dropbox\\ACC Lab\\Nicole Lee"
homecomp = "C:\\Users\\user\\NUS Dropbox\\acclab\\Nicole M Lee"
labcomp = "C:\\Users\\User\\NUS Dropbox\\acclab\\Nicole M Lee"
specifiedpath = labcomp

openPath = specifiedpath + "\\Data Compilation\\Climbing_New\\"
deltagdir = openPath + "Compilation with delta\\2025deltagcollection\\"

for f in font_manager.findSystemFonts(fontpaths=["fonts"]):
    font_manager.fontManager.addfont(f)
plt.rcParams["font.family"] = "Inter"
matplotlib.rcParams['svg.fonttype'] = 'none'

## Load

In [ ]:
files = os.listdir(deltagdir)
totalfile = pd.concat([pd.read_csv(deltagdir + n).assign(responder=n.split(" x ")[1].split("_")[0]) for n in files], ignore_index=True)
totalfile['genotypeandresponder'] = totalfile['MBON'] + "_" + totalfile['responder']

onlymeans = totalfile.loc[:, ~totalfile.columns.str.endswith('_bootstrap')].drop_duplicates().reset_index(drop=True)
mbononly = onlymeans[~onlymeans['MBON'].isin(['R58', 'Th-Gal4', 'R76B09', 'VT999036'])]

counts = mbononly['MBON'].value_counts()
both = mbononly[mbononly['MBON'].isin(counts[counts > 1].index)]

lobelocation = NLProcessing.generate_lobelocation(sorted({n.split(" ")[0] for n in files}), specifiedpath + "\\Data Compilation\\MBONlist.csv")

## Functions

In [ ]:
cols_only = ['MBON', 'responder', 'genotypeandresponder', 'height_deltag', 'speed_deltag', 'bspeed_deltag',
             'maxvelocity_deltag', 'straightindex_deltag', 'meanbout_deltag', 'bout_deltag']

RENAME = {'speed': 'Speed', 'bspeed': 'Bout speed', 'pausepos': 'Pause position', 'bout': '# Bouts', 'meanbout': 'Mean bout time',
          'straightindex': 'Straightness Index', 'height': 'Avg height', 'maxvelocity': 'Max velocity', 'fallnumber_meandiff': '# Fall ΔΔ'}

RESPONDERNAME = {'ACR': 'GtACR1', 'Chrimson2': 'CsChrimson'}

LOBE_PALETTE = {'α': '#E69F00', "α'": '#F0E442', 'β': '#009E73', "β'": '#CC79A7', 'γ': '#999999', 'Calyx': '#000000', 'Unknown': '#FFFFFF'}

NONMETRICS = ['MBON', 'responder', 'genotypeandresponder', 'Lobe', 'Name']


def addlobes(df):
    lobes = lobelocation[['MBON', 'Lobe_location', 'MBON number']].rename(columns={'Lobe_location': 'Lobe', 'MBON number': 'Name'})
    return df.merge(lobes, on='MBON').sort_values(['Name', 'MBON', 'responder'], kind='stable').reset_index(drop=True)


def select(df, cols=cols_only):
    return addlobes(df[cols].rename(columns=lambda c: c.replace('_deltag', '')))


def frame(ax, data):
    for y in (0, len(data)):
        ax.axhline(y=y, color='k', linewidth=5)
    for x in (0, len(data.columns)):
        ax.axvline(x=x, color='k', linewidth=5)


def right_labels(ax, labels):
    twin = ax.twinx()
    twin.set_ylim([0, ax.get_ylim()[0]])
    twin.set_yticks(ax.get_yticks())
    twin.set_yticklabels(labels, fontsize=9)
    for spine in twin.spines.values():
        spine.set_visible(False)


def colorbar_axes(ax):
    return inset_axes(ax, width="15%", height="0.8%", loc='lower right', bbox_to_anchor=(0.12, 1.05, 1, 1), bbox_transform=ax.transAxes, borderpad=-2)


def save(tag):
    plt.savefig(openPath + "images\\" + date + "_" + tag + ".svg", bbox_inches="tight")
    plt.show()


def plot_heatmap(df, title, colormap, tag, paired=False, vmin=-1.5, vmax=1.5):
    rowlabel = 'genotypeandresponder' if paired else 'Name'
    data = df.set_index(rowlabel).drop(columns=[c for c in NONMETRICS if c != rowlabel])
    labelsrc = df.iloc[::2] if paired else df
    labels = (labelsrc['Name' if paired else 'MBON'] + "\n " + labelsrc['Lobe']).tolist()[::-1]

    fig, ax = plt.subplots(figsize=(10, max(4, len(data) * 0.32)))
    sns.set_style("whitegrid", {'axes.grid': False})
    hm = sns.heatmap(data, ax=ax, annot=True, fmt=".1f", vmin=vmin, vmax=vmax, cmap=colormap, clip_on=False,
                     xticklabels=data.rename(columns=RENAME).columns.tolist(), yticklabels=True, annot_kws={"size": 14},
                     cbar_ax=colorbar_axes(ax), cbar_kws=dict(orientation="horizontal", ticks=[vmin, 0, vmax] if vmin < 0 else [vmin, vmax]))
    hm.collections[0].colorbar.ax.tick_params(labelsize=8)
    hm.set_ylabel('')
    hm.set_yticklabels(hm.get_yticklabels(), va='center', rotation=0, fontsize=13)
    hm.set_xticklabels(hm.get_xticklabels(), fontsize=11)
    NLProcessing.wrap_labels(hm, 5)
    hm.tick_params(axis='both', which='both', length=0)
    frame(hm, data)
    right_labels(hm, [x for n in labels for x in ("", n)] if paired else labels)

    fig.suptitle('Plot of MBONs > ' + title + ' and their effects \n across locomotor reactivity parameters for Climbing Assay', x=0.5, y=1.0, weight='bold', fontsize=16)
    fig.tight_layout()
    save(tag + "_deltagheatmapwithlobelocations")


def plot_total_heatmap(df, title, colormap, tag, vmin=-1.5, vmax=1.5):
    falls = df.set_index('MBON')[['fallnumber_meandiff']]
    data = df.set_index('MBON').drop(columns=['responder', 'genotypeandresponder', 'Lobe', 'Name', 'fallnumber_meandiff'])

    fig, (axl, axr) = plt.subplots(1, 2, figsize=(10, max(4, len(data) * 0.32)), gridspec_kw={'width_ratios': [1, 4]})
    sns.set_style("whitegrid", {'axes.grid': False})
    left = sns.heatmap(falls, ax=axl, annot=True, fmt=".1f", vmin=vmin, vmax=vmax, cmap=colormap, clip_on=False,
                       xticklabels=['# falls'], yticklabels=True, annot_kws={"size": 10}, cbar=False)
    right = sns.heatmap(data, ax=axr, annot=True, fmt=".1f", vmin=vmin, vmax=vmax, cmap=colormap, clip_on=False,
                        xticklabels=data.rename(columns=RENAME).columns.tolist(), yticklabels=True, annot_kws={"size": 10},
                        cbar_ax=colorbar_axes(axr), cbar_kws=dict(orientation="horizontal", ticks=[vmin, 0, vmax]))
    right.collections[0].colorbar.ax.tick_params(labelsize=8)

    left.set_ylabel('')
    left.set_xticklabels(left.get_xticklabels(), fontsize=8)
    left.set_title('ΔΔ', fontsize=14)
    right.set_ylabel('')
    right.set_yticklabels([])
    right.set_xticklabels(right.get_xticklabels(), fontsize=8)
    NLProcessing.wrap_labels(right, 5)
    right.set_title('Δg', fontsize=14)
    right.tick_params(axis='both', which='both', length=0)
    frame(left, falls)
    frame(right, data)
    right_labels(right, (df['Name'] + "\n " + df['Lobe']).tolist()[::-1])

    fig.suptitle('Plot of MBONs > ' + title + ' and their effects \n across locomotor reactivity parameters for Climbing Assay', x=0.5, y=1.0, weight='bold', fontsize=16)
    fig.tight_layout()
    save(tag + "_totalheatmapwithlobelocations")


def plot_clustermap(df, title, colormap, tag, paired=False, vmin=-1.5, vmax=1.5, method='complete', metric='euclidean'):
    data = df.set_index('genotypeandresponder').drop(columns=['MBON', 'responder', 'Lobe', 'Name'])
    names = df['Name'] + " (" + df['responder'].map(RESPONDERNAME) + ")" if paired else df['Name']
    lobes = df['Lobe'].apply(NLProcessing.parse_lobes)

    cm = sns.clustermap(data, cmap=colormap, metric=metric, method=method, vmin=vmin, vmax=vmax, col_cluster=False,
                        xticklabels=data.rename(columns=RENAME).columns.tolist(), yticklabels=False,
                        row_colors=['#FFFFFF'] * len(data), dendrogram_ratio=(0.08, 0.08),
                        cbar_kws=dict(orientation='horizontal', ticks=[vmin, vmax]), cbar_pos=(0, 0.87, .02, .1),
                        figsize=(5, max(4, len(data) * 0.22)))
    ax = cm.ax_heatmap
    ax.set_ylabel('')
    ax.set_xticklabels(ax.get_xticklabels(), rotation=0, fontsize=10)
    NLProcessing.wrap_labels(ax, 8)
    cm.fig.suptitle('Clustermap using ' + method + "_" + metric + ' : Δg of MBONs > ' + title + " (Climbing)", weight='bold', fontsize=16, y=1.0)

    gap = 0.008
    rc = cm.ax_row_colors
    p = rc.get_position()
    rc.set_position([p.x0 + gap, p.y0, p.width, p.height])
    h = ax.get_position()
    ax.set_position([h.x0 + gap * 2, h.y0, h.width - gap * 2, h.height])

    order = cm.dendrogram_row.reordered_ind
    rc.clear()
    rc.set_xlim(0, 1)
    rc.set_ylim(0, len(order))
    rc.invert_yaxis()
    rc.axis('off')
    for i, idx in enumerate(order):
        row = lobes.iloc[idx]
        for j, lobe in enumerate(row):
            rc.add_patch(plt.Rectangle((j / len(row), i), 1 / len(row), 1, facecolor=LOBE_PALETTE[lobe], edgecolor='white', linewidth=0.5))
        ax.text(1.005, i + 0.5, names.iloc[idx], ha='left', va='center', fontsize=9, transform=ax.get_yaxis_transform())

    handles = [plt.Rectangle((0, 0), 1, 1, facecolor=c, edgecolor='white', label=l) for l, c in LOBE_PALETTE.items() if l != 'Unknown']
    cm.fig.legend(handles=handles, loc='upper center', bbox_to_anchor=(0.45, 0.95), ncol=len(handles), fontsize=9, frameon=False)
    save(tag + "CLIMBING_clustermap_" + method + "_" + metric)

## 1. ACR

In [ ]:
df = select(mbononly[mbononly['responder'] == "ACR"])
plot_heatmap(df, RESPONDERNAME["ACR"], 'coolwarm', "ACR")
plot_clustermap(df, RESPONDERNAME["ACR"], 'coolwarm', "ACR")

## 2. Chrimson2

In [ ]:
df = select(mbononly[mbononly['responder'] == "Chrimson2"])
plot_heatmap(df, RESPONDERNAME["Chrimson2"], 'coolwarm', "Chrimson2")
plot_clustermap(df, RESPONDERNAME["Chrimson2"], 'coolwarm', "Chrimson2")

## 3. ACR and Chrimson2

In [ ]:
df = select(both)
plot_heatmap(df, "GtACR1 and CsChrimson", 'bwr', "ACRChrimson2", paired=True)
plot_clustermap(df, "GtACR1 and CsChrimson", 'PuOr', "ACRChrimson2", paired=True, method='weighted')

## 4. Specific MBONs

In [ ]:
specificmbons = ["MB083C", "MB112C"]

df = select(both[both['MBON'].isin(specificmbons)])
plot_heatmap(df, "GtACR1 and CsChrimson", 'bwr', "ACRChrimson2_specific", paired=True)
plot_clustermap(df, "GtACR1 and CsChrimson", 'PuOr', "ACRChrimson2_specific", paired=True, method='weighted')

## 5. Differenced

In [ ]:
grouped = select(both).drop(columns=['responder', 'genotypeandresponder', 'Lobe', 'Name']).groupby('MBON')
diff = (grouped.max() - grouped.min()).reset_index()
df = addlobes(diff.assign(responder='diff', genotypeandresponder=diff['MBON'])).sort_values('Lobe', kind='stable').reset_index(drop=True)
plot_heatmap(df, "abs difference, CsChrimson vs GtACR1", 'Greens', "absdiff", vmin=0, vmax=1.5)
plot_clustermap(df, "abs difference, CsChrimson vs GtACR1", 'Blues', "absdiff", vmin=0, vmax=1.5, method='weighted')

## 6. Total metrics

In [ ]:
responder = "ACR"

df = select(mbononly[mbononly['responder'] == responder], cols_only + ['pausepos_deltag', 'fallnumber_meandiff'])
plot_total_heatmap(df, RESPONDERNAME[responder], 'BrBG' if responder == "ACR" else 'RdGy_r', responder + "_total")
plot_clustermap(df, RESPONDERNAME[responder], 'BrBG' if responder == "ACR" else 'RdGy', responder + "_total", method='weighted')